# Deep Convolutional Q-Learning for Pac-Man

## Part 0 - Installing the required packages and importing the libraries

### Installing Gymnasium

In [10]:
!pip install gymnasium
!pip install "gymnasium[atari, accept-rom-license]"
!apt-get install -y swig
!pip install gymnasium[box2d]

"apt-get" non � riconosciuto come comando interno o esterno,
 un programma eseguibile o un file batch.


### Importing the libraries

In [11]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
from torch.utils.data import DataLoader, TensorDataset

## Part 1 - Building the AI

### Creating the architecture of the Neural Network

In [12]:
class Network(nn.Module):
    def __init__(self, action_size, seed=42):
        super(Network, self).__init__()
        self.seed = torch.manual_seed(seed)
        # Input: 1x84x84 (grayscale)
        self.conv1 = nn.Conv2d(1, 32, kernel_size=8, stride=4)  # -> 32x20x20
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2) # -> 64x9x9
        self.bn2   = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1) # -> 64x7x7
        self.bn3   = nn.BatchNorm2d(64)
        self.fc1   = nn.Linear(64 * 7 * 7, 512)
        self.fc2   = nn.Linear(512, action_size)

    def forward(self, state):
        x = F.relu(self.bn1(self.conv1(state)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

## Part 2 - Training the AI

### Setting up the environment

In [13]:
import gymnasium as gym
import ale_py
gym.register_envs(ale_py)
env = gym.make('ALE/MsPacman-v5', full_action_space=False, repeat_action_probability=0.0)
state_shape = env.observation_space.shape
state_size = env.observation_space.shape[0]
number_actions = env.action_space.n
print('State shape:', state_shape)
print('State size:', state_size)
print('Number of actions:', number_actions)

State shape: (210, 160, 3)
State size: 210
Number of actions: 9


### Initializing the hyperparameters

In [14]:
learning_rate = 5e-4
minibatch_size = 64
discount_factor = 0.99

### Preprocessing the frames

In [15]:
from PIL import Image
from torchvision import transforms

def preprocess_state(frame):
    frame = Image.fromarray(frame).convert('L')  # grayscale
    preprocess = transforms.Compose([
        transforms.Resize((84, 84)),
        transforms.ToTensor(),
    ])
    return preprocess(frame).unsqueeze(0)  # (1, 1, 84, 84)

### Implementing the DCQN class

In [16]:
LEARN_EVERY  = 4     # learn ogni N step
UPDATE_EVERY = 1000  # hard-copy local -> target ogni N step

class Agent():

    def __init__(self, action_size):
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.action_size = action_size
        self.local_qnetwork  = Network(action_size).to(self.device)
        self.target_qnetwork = Network(action_size).to(self.device)
        self.optimizer = optim.Adam(self.local_qnetwork.parameters(), lr=learning_rate)
        self.memory = deque(maxlen=100000)
        self.t_step = 0

    def step(self, state, action, reward, next_state, done):
        self.memory.append((preprocess_state(state),
                            action,
                            reward,
                            preprocess_state(next_state),
                            done))
        self.t_step += 1
        if self.t_step % LEARN_EVERY == 0 and len(self.memory) > minibatch_size:
            experiences = random.sample(self.memory, k=minibatch_size)
            self.learn(experiences, discount_factor)
        if self.t_step % UPDATE_EVERY == 0:
            self._update_target_network()

    def act(self, state, epsilon=0.):
        state = preprocess_state(state).to(self.device)
        self.local_qnetwork.eval()
        with torch.no_grad():
            action_values = self.local_qnetwork(state)
        self.local_qnetwork.train()
        if random.random() > epsilon:
            return np.argmax(action_values.cpu().data.numpy())
        else:
            return random.choice(np.arange(self.action_size))

    def learn(self, experiences, discount_factor):
        states, actions, rewards, next_states, dones = zip(*experiences)
        states      = torch.cat(states).float().to(self.device)
        actions     = torch.from_numpy(np.vstack(actions)).long().to(self.device)
        rewards     = torch.from_numpy(np.vstack(rewards)).float().to(self.device)
        next_states = torch.cat(next_states).float().to(self.device)
        dones       = torch.from_numpy(np.vstack(dones).astype(np.uint8)).float().to(self.device)

        # Double DQN: local selects action, target evaluates it
        next_actions   = self.local_qnetwork(next_states).detach().argmax(1).unsqueeze(1)
        next_q_targets = self.target_qnetwork(next_states).detach().gather(1, next_actions)
        q_targets  = rewards + discount_factor * next_q_targets * (1 - dones)
        q_expected = self.local_qnetwork(states).gather(1, actions)

        loss = F.mse_loss(q_expected, q_targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def _update_target_network(self):
        self.target_qnetwork.load_state_dict(self.local_qnetwork.state_dict())

### Initializing the DCQN agent

In [17]:
agent = Agent(number_actions)

### Training the DCQN agent

In [18]:
number_episodes = 2000
maximum_number_timesteps_per_episode = 10000
epsilon_starting_value = 1.0
epsilon_ending_value   = 0.01
epsilon_decay_value    = 0.995
epsilon = epsilon_starting_value
scores_on_100_episodes = deque(maxlen=100)

for episode in range(1, number_episodes + 1):
    state, _ = env.reset()
    score = 0
    for t in range(maximum_number_timesteps_per_episode):
        action = agent.act(state, epsilon)
        next_state, reward, done, _, _ = env.step(action)
        agent.step(state, action, reward, next_state, done)
        state = next_state
        score += reward
        if done:
            break
    scores_on_100_episodes.append(score)
    epsilon = max(epsilon_ending_value, epsilon_decay_value * epsilon)
    print('\rEpisode {}\tAverage Score: {:.2f}'.format(episode, np.mean(scores_on_100_episodes)), end="")
    if episode % 100 == 0:
        print('\rEpisode {}\tAverage Score: {:.2f}'.format(episode, np.mean(scores_on_100_episodes)))
    if np.mean(scores_on_100_episodes) >= 1500.0:
        print('\nEnvironment solved in {:d} episodes!\tAverage Score: {:.2f}'.format(episode - 100, np.mean(scores_on_100_episodes)))
        torch.save(agent.local_qnetwork.state_dict(), 'checkpoint.pth')
        break

Episode 100	Average Score: 296.10
Episode 200	Average Score: 495.20
Episode 300	Average Score: 626.20
Episode 400	Average Score: 726.60
Episode 500	Average Score: 876.50
Episode 600	Average Score: 999.100
Episode 700	Average Score: 1199.80
Episode 800	Average Score: 1171.40
Episode 900	Average Score: 1273.40
Episode 1000	Average Score: 1289.20
Episode 1061	Average Score: 1503.80
Environment solved in 961 episodes!	Average Score: 1503.80


## Part 3 - Visualizing the results

In [19]:
import glob
import io
import base64
import imageio
from IPython.display import HTML, display

def show_video_of_model(agent, env_name):
    env = gym.make(env_name, render_mode='rgb_array')
    state, _ = env.reset()
    done = False
    frames = []
    while not done:
        frame = env.render()
        frames.append(frame)
        action = agent.act(state)
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
    env.close()
    imageio.mimsave('video.mp4', frames, fps=30)

show_video_of_model(agent, 'ALE/MsPacman-v5')

def show_video():
    mp4list = glob.glob('*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

show_video()

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (160, 210) to (160, 224) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
